# CatBoost + Optuna with Greedy Feature Removal

**Goal:** find the smallest, highest-scoring feature subset by iteratively dropping the least-important column and re-tuning hyperparameters from scratch.

**Algorithm:**
1. Tune CatBoost on the **full** feature set with Optuna (`N_TRIALS_PER_ATTEMPT` trials) using 5-fold StratifiedKFold CV ROC-AUC.
2. Rank features by importance (ascending = least important first).
3. **Round loop:** walk down the ranking. For each candidate column:
    - Drop it, re-run a full Optuna tune on the smaller feature set.
    - If new CV ROC-AUC beats current by **more than `IMPROVEMENT_THRESHOLD`** → keep the removal, recompute feature importance, start a new round.
    - Otherwise → restore the column, try the next least-important one.
4. When an entire round produces no qualifying improvement, stop.
5. Re-tune once more on the winning feature set with a larger trial budget (`N_TRIALS_FINAL`).
6. Retrain on the **full training set** and write boolean predictions to `submission.csv`.

**Every attempt is appended to `feature_selection_log.md` as it happens**, so if the run is interrupted you still have a record.

**⚠️ Runtime warning:** Each Optuna attempt does `N_TRIALS_PER_ATTEMPT × 5` CatBoost fits. With ~60 features and 25 trials/attempt, one round can be hours. Lower `N_TRIALS_PER_ATTEMPT` to speed things up if needed.

## 0. Config and imports

In [1]:
TRAIN_PATH = "/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_train_cleaned.csv"
TEST_PATH  = "/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_test_cleaned.csv"
LOG_PATH = "feature_selection_log.md"
SUBMISSION_PATH = "submission.csv"

N_TRIALS_PER_ATTEMPT = 25      # Optuna trials per feature-removal attempt
N_TRIALS_FINAL       = 50      # Optuna trials for the final re-tune on the winning feature set
IMPROVEMENT_THRESHOLD = 0.002  # CV ROC-AUC must improve by more than this to keep a removal
CV_SPLITS = 5
RANDOM_SEED = 42

import json
from datetime import datetime
import numpy as np
import pandas as pd
import optuna
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Load training data

In [2]:
train = pd.read_csv(TRAIN_PATH)
DROP_COLS = ["GuestID", "Churned"]
X_full = train.drop(columns=DROP_COLS)
y = train["Churned"]

neg, pos = int((y == 0).sum()), int((y == 1).sum())
SCALE_POS_WEIGHT = neg / pos

print(f"Training data: {X_full.shape[0]} rows, {X_full.shape[1]} features")
print(f"Class balance: pos={pos}, neg={neg}, scale_pos_weight={SCALE_POS_WEIGHT:.4f}")

Training data: 6954 rows, 59 features
Class balance: pos=3502, neg=3452, scale_pos_weight=0.9857


## 2. Helper functions

- `make_objective`: builds the Optuna objective for the given (X, y), using 5-fold StratifiedKFold CV ROC-AUC on the model's hard 0/1 predictions (matches the boolean target the submission uses).
- `run_optuna`: runs `n_trials` of Optuna and returns `(best_cv_score, best_params, feature_importance_df)`. Feature importance is computed by refitting on a holdout split with the best params.
- `log_md`: appends a line to the markdown log AND prints it so you can follow progress live.

In [3]:
def make_objective(X, y, cat_features):
    def objective(trial):
        params = {
            "iterations": 1000,
            "depth": trial.suggest_int("depth", 4, 10),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 100),
            "scale_pos_weight": SCALE_POS_WEIGHT,
            "eval_metric": "AUC",
            "early_stopping_rounds": 50,
            "random_seed": RANDOM_SEED,
            "verbose": False,
        }
        skf = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_SEED)
        scores = []
        for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            model = CatBoostClassifier(**params)
            model.fit(
                Pool(X_tr, y_tr, cat_features=cat_features),
                eval_set=Pool(X_va, y_va, cat_features=cat_features),
            )
            preds = model.predict(X_va)
            scores.append(roc_auc_score(y_va, preds))
            trial.report(float(np.mean(scores)), fold_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return float(np.mean(scores))
    return objective


def run_optuna(X, y, n_trials):
    """Tune CatBoost on (X, y). Returns (best_cv_score, best_params, feature_importance_df)."""
    cat_features = [c for c in X.columns if X[c].dtype == "object"]
    X_local = X.copy()
    if cat_features:
        X_local[cat_features] = X_local[cat_features].fillna("missing")

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),
    )
    study.optimize(
        make_objective(X_local, y, cat_features),
        n_trials=n_trials,
        show_progress_bar=False,
    )

    best_params = {
        **study.best_params,
        "iterations": 1000,
        "scale_pos_weight": SCALE_POS_WEIGHT,
        "eval_metric": "AUC",
        "early_stopping_rounds": 50,
        "random_seed": RANDOM_SEED,
        "verbose": False,
    }

    X_tr, X_va, y_tr, y_va = train_test_split(
        X_local, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
    )
    fi_model = CatBoostClassifier(**best_params)
    fi_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    fi_model.fit(fi_pool, eval_set=Pool(X_va, y_va, cat_features=cat_features))
    fi = pd.DataFrame({
        "feature": X_local.columns,
        "importance": fi_model.get_feature_importance(fi_pool),
    }).sort_values("importance", ascending=True).reset_index(drop=True)

    return float(study.best_value), best_params, fi


def _tunable(params):
    fixed = {"iterations", "scale_pos_weight", "eval_metric", "early_stopping_rounds", "random_seed", "verbose"}
    return {k: (round(v, 6) if isinstance(v, float) else v) for k, v in params.items() if k not in fixed}


def log_md(msg=""):
    with open(LOG_PATH, "a") as f:
        f.write(msg + "\n")
    print(msg)

## 3. Baseline: tune on the full feature set

This also resets the log file.

In [4]:
with open(LOG_PATH, "w") as f:
    f.write("# Feature Selection Log\n\n")
    f.write(f"**Started:** {datetime.now().isoformat(timespec='seconds')}\n\n")
    f.write("## Config\n\n")
    f.write(f"- `N_TRIALS_PER_ATTEMPT`: {N_TRIALS_PER_ATTEMPT}\n")
    f.write(f"- `N_TRIALS_FINAL`: {N_TRIALS_FINAL}\n")
    f.write(f"- `IMPROVEMENT_THRESHOLD`: {IMPROVEMENT_THRESHOLD}\n")
    f.write(f"- `CV_SPLITS`: {CV_SPLITS}\n")
    f.write(f"- `RANDOM_SEED`: {RANDOM_SEED}\n")
    f.write(f"- `SCALE_POS_WEIGHT`: {SCALE_POS_WEIGHT:.6f}\n")
    f.write(f"- Starting features ({X_full.shape[1]}): {list(X_full.columns)}\n\n")

log_md(f"## Baseline (all {X_full.shape[1]} features)\n")
baseline_score, baseline_params, baseline_fi = run_optuna(X_full, y, N_TRIALS_PER_ATTEMPT)
log_md(f"- **Baseline CV ROC-AUC:** `{baseline_score:.5f}`")
log_md(f"- Best params: `{json.dumps(_tunable(baseline_params))}`")
log_md("\n### Initial feature importance (ascending — least important first):\n")
log_md("```")
log_md(baseline_fi.to_string(index=False))
log_md("```\n")

## Baseline (all 59 features)

- **Baseline CV ROC-AUC:** `0.82114`
- Best params: `{"depth": 8, "learning_rate": 0.013365, "l2_leaf_reg": 3.741304, "bagging_temperature": 0.761061, "random_strength": 0.442185, "border_count": 247, "min_data_in_leaf": 14}`

### Initial feature importance (ascending — least important first):

```
                   feature  importance
               RoomFloor_T    0.000000
  ReferralSource_Newspaper    0.000000
  ReferralSource_Pinterest    0.002052
   ReferralSource_LinkedIn    0.002739
               RoomFloor_D    0.008163
   ReferralSource_Magazine    0.011606
         RoomFloor_Unknown    0.015727
                SharedRoom    0.016172
          RoomSide_Unknown    0.020169
      ReferralSource_Radio    0.021086
         ReferralSource_TV    0.022668
          AgeGroup_Unknown    0.033597
                       VIP    0.034758
      ReferralSource_Flyer    0.046729
     PackageType_NoPackage    0.067658
            Region_Unknown    0.083768
    Re

## 4. Greedy column removal

Walk down the importance ranking. Drop → retune → compare. Accept if Δ > `IMPROVEMENT_THRESHOLD`. Restart the round on every acceptance (new model → new importance ordering).

In [ ]:
current_X = X_full.copy()
current_score = baseline_score
current_params = baseline_params
current_fi = baseline_fi
removed_cols = []
round_idx = 0

while True:
    round_idx += 1
    log_md(f"## Round {round_idx} (features: {current_X.shape[1]}, current score: {current_score:.5f})\n")
    candidates = current_fi["feature"].tolist()  # ascending importance

    log_md("**10 least-important features this round (re-ranked from current model):**\n")
    log_md("```")
    log_md(current_fi.head(10).to_string(index=False))
    log_md("```\n")

    improved = False
    for candidate in candidates:
        log_md(f"### Round {round_idx} — try removing `{candidate}`")
        trial_X = current_X.drop(columns=[candidate])
        trial_score, trial_params, trial_fi = run_optuna(trial_X, y, N_TRIALS_PER_ATTEMPT)
        delta = trial_score - current_score
        status = "✅ ACCEPT" if delta > IMPROVEMENT_THRESHOLD else "❌ REJECT"
        log_md(f"- New CV ROC-AUC: `{trial_score:.5f}` (Δ = `{delta:+.5f}`) → **{status}**")

        if delta > IMPROVEMENT_THRESHOLD:
            removed_cols.append(candidate)
            current_X = trial_X
            current_score = trial_score
            current_params = trial_params
            current_fi = trial_fi
            improved = True
            log_md(f"- Dropped `{candidate}`. Remaining features: {current_X.shape[1]}.")
            log_md(f"- New best params: `{json.dumps(_tunable(current_params))}`\n")
            break
        else:
            log_md(f"- Kept `{candidate}`. Trying next column.\n")

    if not improved:
        log_md(f"## Stop — round {round_idx} produced no improvement > {IMPROVEMENT_THRESHOLD}.\n")
        break

log_md("## Search summary\n")
log_md(f"- **Final CV ROC-AUC (during search):** `{current_score:.5f}` (baseline `{baseline_score:.5f}`, Δ = `{current_score - baseline_score:+.5f}`)")
log_md(f"- Removed {len(removed_cols)} columns in order: `{removed_cols}`")
log_md(f"- Kept {current_X.shape[1]} columns: `{list(current_X.columns)}`\n")

## 5. Final re-tune on the winning feature set

Larger trial budget. If it beats the search result, use it; otherwise fall back to the search result's params.

In [ ]:
log_md(f"## Final re-tune ({N_TRIALS_FINAL} trials on winning feature set)\n")
final_score, final_params, _ = run_optuna(current_X, y, N_TRIALS_FINAL)
log_md(f"- Re-tuned CV ROC-AUC: `{final_score:.5f}` (search result was `{current_score:.5f}`)")
log_md(f"- Re-tuned best params: `{json.dumps(_tunable(final_params))}`\n")

if final_score > current_score:
    winning_params = final_params
    winning_score = final_score
    log_md("✅ Re-tune improved on the search result — using re-tuned params for the submission.")
else:
    winning_params = current_params
    winning_score = current_score
    log_md("ℹ️ Re-tune did not beat the search result — falling back to search-result params for the submission.")

log_md(f"\n- **Winning features ({current_X.shape[1]}):** `{list(current_X.columns)}`")
log_md(f"- **Winning CV ROC-AUC:** `{winning_score:.5f}`")
log_md(f"- **Winning params:** `{json.dumps(_tunable(winning_params))}`\n")

## 6. Build the submission

Load the test set, drop the same columns that lost the search, train one final CatBoost on the **full** training data with the winning params, predict, and write boolean labels to `submission.csv`.

In [ ]:
test = pd.read_csv(TEST_PATH)
submission = test[["GuestID"]].copy()

winning_features = list(current_X.columns)
X_test = test[winning_features].copy()
X_train_final = current_X.copy()

cat_features = [c for c in X_train_final.columns if X_train_final[c].dtype == "object"]
if cat_features:
    X_train_final[cat_features] = X_train_final[cat_features].fillna("missing")
    X_test[cat_features] = X_test[cat_features].fillna("missing")

final_model = CatBoostClassifier(**winning_params)
final_model.fit(Pool(X_train_final, y, cat_features=cat_features))

preds = final_model.predict(X_test).flatten().astype(bool)
submission["Churned"] = preds

submission.to_csv(SUBMISSION_PATH, index=False)

print(submission.head())
print(submission["Churned"].value_counts())
print(f"\nSaved {len(submission)} predictions to {SUBMISSION_PATH}")

log_md("## Submission\n")
log_md(f"- Wrote `{SUBMISSION_PATH}` ({len(submission)} rows)")
log_md(f"- Predictions: {int(submission['Churned'].sum())} True, {int((~submission['Churned']).sum())} False")
log_md(f"\n**Completed:** {datetime.now().isoformat(timespec='seconds')}\n")